In [ ]:
!pip install -e ../.

In [1]:
import json
import pandas as pd

In [ ]:
json_snapshot = "/Users/bjoern/git/Reanimator/data/arxiv-metadata-oai-snapshot.json"

records = []
with open(json_snapshot, 'r') as f:
    for i, line in enumerate(f):

        paper = json.loads(line)

        records.append({
            'id': paper['id'],
            'title': paper['title'],
            'abstract': paper['abstract'],
            'categories': paper['categories'],
            'date': paper['update_date'],
            'doi': paper['doi']       })

# Convert to DataFrame

df = pd.DataFrame(records)
df.dropna(subset=['abstract'], inplace=True)

In [13]:
sample_ids = pd.read_csv('/Users/bjoern/git/Reanimator/notebooks/arxix_ids.txt', header=None, dtype=str, keep_default_na=False).iloc[:, 0].tolist()

In [14]:
sample_ids

['2403.08173',
 '2312.08156',
 '2407.13393',
 '2401.10883',
 '2505.09600',
 '2304.12294',
 '2503.07005',
 '2506.11170',
 '2410.21590',
 '2506.10743',
 '2502.11846',
 '2410.07244',
 '2010.12490',
 '2502.17595',
 '2503.14986',
 '2502.11709',
 '2405.12730',
 '2503.15604',
 '2508.10771',
 '2411.04955',
 '2503.23807',
 '2508.03497',
 '2411.09177',
 '2401.12913',
 '2504.16847',
 '2504.17863',
 '2506.11224',
 '2407.17490',
 '2208.08330',
 '2409.03900',
 '2506.11217',
 '1807.07780',
 '2409.08803',
 '2502.03812',
 '2505.23243',
 '2310.06026',
 '2311.18706',
 '2505.11617',
 '2506.06159',
 '2405.07246',
 '2409.08127',
 '2211.16119',
 '2408.08555',
 '2407.20368',
 '2401.05202',
 '2305.08384',
 '2411.18396',
 '2210.15660',
 '2508.10025',
 '2406.06259',
 '2506.04385',
 '2306.11348',
 '2505.13987',
 '2006.05686',
 '2505.14003',
 '2307.15845',
 '2412.08017',
 '2309.16912',
 '2406.07497',
 '2406.00863',
 '2302.11617',
 '2506.23879',
 '2310.01555',
 '2210.17288',
 '2410.11035',
 '2401.01999',
 '2305.077

In [15]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler, TopicChunkPair, calculate_cohens_kappa
from reanimator.retrieval import Indexer, Retriever, reciprocal_rank_fusion, run_experiment
from reanimator.models import save_judgements, load_judgements

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
import nltk

load_dotenv()

import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/bjoern/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
reanimator = Reanimator(arxiv_ids=sample_ids)


INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


In [17]:
docs = reanimator.load_documents()

Step 1: Loading documents from source...


In [18]:
reanimator.download_documents(docs)


Step 2: Fetching URLs and downloading PDFs...
All DOIs already have cached URLs.


PDF downloading complete.


In [19]:
docs

[Document(doc_id='2403.08173', doi=None, url='https://arxiv.org/pdf/2403.08173', pdf_path='data/pdfs/2403.08173.pdf', text=None, tables=[], figures=[], chunks=[], metadata={'source': 'arxiv', 'arxiv_id': '2403.08173'}),
 Document(doc_id='2312.08156', doi=None, url='https://arxiv.org/pdf/2312.08156', pdf_path='data/pdfs/2312.08156.pdf', text=None, tables=[], figures=[], chunks=[], metadata={'source': 'arxiv', 'arxiv_id': '2312.08156'}),
 Document(doc_id='2407.13393', doi=None, url='https://arxiv.org/pdf/2407.13393', pdf_path='data/pdfs/2407.13393.pdf', text=None, tables=[], figures=[], chunks=[], metadata={'source': 'arxiv', 'arxiv_id': '2407.13393'}),
 Document(doc_id='2401.10883', doi=None, url='https://arxiv.org/pdf/2401.10883', pdf_path='data/pdfs/2401.10883.pdf', text=None, tables=[], figures=[], chunks=[], metadata={'source': 'arxiv', 'arxiv_id': '2401.10883'}),
 Document(doc_id='2505.09600', doi=None, url='https://arxiv.org/pdf/2505.09600', pdf_path='data/pdfs/2505.09600.pdf', te

In [20]:
accelerator_options = AcceleratorOptions(
        num_threads=8, device=AcceleratorDevice.MPS
    )

In [ ]:
reanimator.extract_content(docs)
reanimator.save_documents(docs, "documents")


Step 3: Extracting content from PDFs...


Extracting Content:   1%|          | 1/100 [00:49<1:20:56, 49.05s/it]/Users/bjoern/git/Reanimator/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/bjoern/git/Reanimator/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/bjoern/git/Reanimator/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/bjoern/git/Reanimator/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then devic